In [12]:
import json
import pandas as pd

# -------------------------
# 1. โหลดไฟล์
# -------------------------
with open("../commentMyanimelist/comment.json", "r", encoding="utf-8") as f:
    comment_data = json.load(f)

with open("../commentYoutube/anime_comments_cleaned_ultra_promax.json", "r", encoding="utf-8") as f:
    youtube_data = json.load(f)

with open("../commentAnilist/cleaned_anime_comments_pro.json", "r", encoding="utf-8") as f:
    pro_data = json.load(f)

with open("../animelist/anime_2025_by_season.json", "r", encoding="utf-8") as f:
    anime_data = json.load(f)

# -------------------------
# 2. สร้าง DataFrame และ Rename Columns
# -------------------------
df_comment = pd.DataFrame(comment_data).rename(columns={
    "title": "anime_name",
    "text_c": "comment",
    "uidpost": "malID"
})

df_youtube = pd.DataFrame(youtube_data)

df_pro = pd.DataFrame(pro_data).rename(columns={
    "mal_id": "malID",
    "anime": "anime_name",
    "user": "user_c",
    "text": "comment"
})

# เลือกเฉพาะ Column ที่ต้องการ
cols = ["malID", "anime_name", "user_c", "comment"]
df_comment = df_comment[cols]
df_youtube = df_youtube[cols]
df_pro = df_pro[cols]

# -------------------------
# 3. รวม Dataset หลัก (Concat)
# -------------------------
df_all = pd.concat([df_comment, df_youtube, df_pro], ignore_index=True)

# แปลง malID เป็น string เพื่อป้องกันปัญหา Type mismatch ตอน merge
df_all['malID'] = df_all['malID'].astype(str)

print(f"Total rows before merge: {len(df_all)}") # ควรได้ 80076

# -------------------------
# 4. เตรียมข้อมูล Genres (ป้องกันแถวซ้ำ)
# -------------------------
anime_list = []
for season, animes in anime_data["seasons"].items():
    for anime in animes:
        genres = [g["name"] for g in anime.get("genres", [])]
        anime_list.append({
            "malID": str(anime["mal_id"]), # แปลงเป็น string ทันที
            "Genres": ", ".join(genres)
        })

df_genres = pd.DataFrame(anime_list)

# *** จุดสำคัญ: ลบ malID ที่ซ้ำในไฟล์ anime_data ออกก่อน ***
# เพราะถ้า 1 malID มีมากกว่า 1 แถวใน df_genres พอมารวมกับ df_all แถวจะงอกครับ
df_genres = df_genres.drop_duplicates(subset=['malID'], keep='first')

# -------------------------
# 5. Merge Genres เข้ากับ df_all
# -------------------------
# ใช้ how="left" เพื่อรักษาคอมเมนต์ทุกอันไว้ แม้เรื่องนั้นจะไม่มีข้อมูล Genres ก็ตาม
df_all = df_all.merge(df_genres, on="malID", how="left")



Total rows before merge: 80076


In [13]:
# -------------------------
# 6. จัดการ ID และบันทึก
# -------------------------
# สร้าง ID ใหม่หลังจาก merge เสร็จแล้วเพื่อความเรียงตัวของลำดับ
if 'ID' in df_all.columns:
    df_all = df_all.drop(columns=['ID'])
df_all.insert(0, "ID", range(1, len(df_all) + 1))

print("--- Info ---")
print(df_all.info())
print(f"Final row count: {len(df_all)}") # ตอนนี้ควรจะเท่ากับ 80076 แล้ว

# แสดงตัวอย่าง
display(df_all.head())

--- Info ---
<class 'pandas.DataFrame'>
RangeIndex: 80076 entries, 0 to 80075
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   ID          80076 non-null  int64
 1   malID       80076 non-null  str  
 2   anime_name  80076 non-null  str  
 3   user_c      80076 non-null  str  
 4   comment     80076 non-null  str  
 5   Genres      80076 non-null  str  
dtypes: int64(1), str(5)
memory usage: 3.7 MB
None
Final row count: 80076


,ID,malID,anime_name,user_c,comment,Genres
0,1,58567,Ore dake Level Up na Ken Season 2: Arise from ...,Tkit,This is genuinely the most pathetic new mainst...,"Action, Adventure, Fantasy"
1,2,58567,Ore dake Level Up na Ken Season 2: Arise from ...,Marinate1016,No amount of memes or social media posts could...,"Action, Adventure, Fantasy"
2,3,58567,Ore dake Level Up na Ken Season 2: Arise from ...,keirashii,This review contains spoilers. If we were to f...,"Action, Adventure, Fantasy"
3,4,58567,Ore dake Level Up na Ken Season 2: Arise from ...,Bardwyne,"Just as the last season, Solo Leveling s2 has ...","Action, Adventure, Fantasy"
4,5,58567,Ore dake Level Up na Ken Season 2: Arise from ...,ShishiKami,It's honestly just as bad as the previous seas...,"Action, Adventure, Fantasy"


In [14]:
# -------------------------
# ตรวจสอบ Genres ที่ว่าง (Missing Genres)
# -------------------------

# 1. กรองเฉพาะแถวที่ Genres เป็นค่าว่าง (NaN)
df_missing_genres = df_all[df_all["Genres"].isna()]

# 2. นับจำนวนแถวที่ Genres ว่าง
missing_count = len(df_missing_genres)
print(f"จำนวนคอมเมนต์ที่หา Genres ไม่เจอ: {missing_count} แถว")

# 3. ดูว่ามี malID ไหนบ้างที่ข้อมูลหาย (เอาเฉพาะตัวที่ไม่ซ้ำมาดู)
missing_malids = df_missing_genres["malID"].unique()
print(f"จำนวน malID ที่ข้อมูล Genres ขาดไป: {len(missing_malids)} รายการ")

# 4. แสดงตัวอย่าง malID และชื่ออนิเมะที่ไม่มี Genres
if missing_count > 0:
    print("\nตัวอย่างอนิเมะที่ไม่มีข้อมูล Genres:")
    print(df_missing_genres[["malID", "anime_name"]].drop_duplicates().head(10))

# 5. (Optional) บันทึกรายชื่อที่ Genres ว่างออกมาตรวจสอบ
# df_missing_genres.to_csv("missing_genres_check.csv", index=False, encoding="utf-8")

จำนวนคอมเมนต์ที่หา Genres ไม่เจอ: 0 แถว
จำนวน malID ที่ข้อมูล Genres ขาดไป: 0 รายการ


In [15]:
print("--- Final Info ---")
print(df_all.info())
print(f"Final row count: {len(df_all)}") # ตอนนี้ควรจะเท่ากับ 80076 แล้ว

# แสดงตัวอย่าง
display(df_all.head())

# Save
df_all.to_csv("anime_comments_all.csv", index=False, encoding="utf-8")

--- Final Info ---
<class 'pandas.DataFrame'>
RangeIndex: 80076 entries, 0 to 80075
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   ID          80076 non-null  int64
 1   malID       80076 non-null  str  
 2   anime_name  80076 non-null  str  
 3   user_c      80076 non-null  str  
 4   comment     80076 non-null  str  
 5   Genres      80076 non-null  str  
dtypes: int64(1), str(5)
memory usage: 3.7 MB
None
Final row count: 80076


,ID,malID,anime_name,user_c,comment,Genres
0,1,58567,Ore dake Level Up na Ken Season 2: Arise from ...,Tkit,This is genuinely the most pathetic new mainst...,"Action, Adventure, Fantasy"
1,2,58567,Ore dake Level Up na Ken Season 2: Arise from ...,Marinate1016,No amount of memes or social media posts could...,"Action, Adventure, Fantasy"
2,3,58567,Ore dake Level Up na Ken Season 2: Arise from ...,keirashii,This review contains spoilers. If we were to f...,"Action, Adventure, Fantasy"
3,4,58567,Ore dake Level Up na Ken Season 2: Arise from ...,Bardwyne,"Just as the last season, Solo Leveling s2 has ...","Action, Adventure, Fantasy"
4,5,58567,Ore dake Level Up na Ken Season 2: Arise from ...,ShishiKami,It's honestly just as bad as the previous seas...,"Action, Adventure, Fantasy"
